In [1]:
partition = 100

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [8, 9, 10, 11, 12, 13]
hidden_dim = [1024, 768]
batch_size_values = [256, 512]
tree_feature_rates = [0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2]
lrs = [0.001, 0.01]

n_iter = 150
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0

sampled_configs = random.sample(param_space, min(n_iter, len(param_space)))
i = 1
for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")

    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i =i + 1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(best_acc)
print(f"Best accuracy: {best_acc}")



Running: n_tree=20, t_depth=10, hd=1024, batch_size=512, feature_rate=0.2, dropout=0.2, lr=0.001
1 / 100
Use gtd100 dataset
Patience: 100


Training Epochs:   8%|▊         | 34/400 [00:04<00:49,  7.42it/s]


KeyboardInterrupt: 

In [ ]:
#Running: n_tree=50, t_depth=10, hd=768, batch_size=256, feature_rate=0.1, dropout=0.1, lr=0.01
# Bästa innan 07-10 acc: 0.540476


In [ ]:
"""

========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd100
  Hidden Dim: 768
  n_tree: 50, tree_depth: 8, tree_feature_rate: 0.1
  Batch size: 256, Dropout: 0.1, LR: 0.01

Best Accuracy: 0.4311
Weighted Precision: 0.4136, Recall: 0.4311, F1 Score: 0.4068, ROCAUC: 0.9071
Macro Precision: 0.4136, Recall: 0.4311, F1 Score: 0.4068, ROCAUC: 0.9071
Micro Precision: 0.4311, Recall: 0.4311, F1 Score: 0.4311, ROCAUC: 0.9137
"""

In [ ]:
"""Best hyperparameter configuration:
{'n_tree': 10, 'tree_depth': 11, 'batch_size': 256, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'lr': 0.01}
0.5285714285714286
Best accuracy: 0.5285714285714286"""

"Best hyperparameter configuration:\n{'n_tree': 10, 'tree_depth': 11, 'batch_size': 256, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'lr': 0.01}\n0.5285714285714286\nBest accuracy: 0.5285714285714286"

In [ ]:
"""sys.argv = [
    'train.py',
    '-dataset', f'gtd{partition}',
    '-n_class', '30',
    '-gpuid', '0',
    '-n_tree', str(best_config['n_tree']),
    '-tree_depth', str(best_config['tree_depth']),
    '-batch_size', str(best_config['batch_size']),
    '-epochs', '1000',
    '-verbose', '1',
    '-jointly_training'
]"""

sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()


Use gtd100 dataset
Patience: 300


Training Epochs:   3%|▎         | 50/1500 [00:18<08:55,  2.71it/s]

[Epoch 50] Train Loss: 1.4980, Eval Loss: 1.9419, Eval Accuracy: 0.4524


Training Epochs:   7%|▋         | 100/1500 [00:37<08:22,  2.78it/s]

[Epoch 100] Train Loss: 1.2262, Eval Loss: 1.8435, Eval Accuracy: 0.4786


...

[Epoch 450] Train Loss: 1.0621, Eval Loss: 1.9390, Eval Accuracy: 0.5024


Training Epochs:  33%|███▎      | 500/1500 [03:03<06:05,  2.74it/s]

[Epoch 500] Train Loss: 1.0604, Eval Loss: 1.9516, Eval Accuracy: 0.4976


Training Epochs:  35%|███▍      | 519/1500 [03:10<06:00,  2.72it/s]

Early stopping at epoch 520
Evaluating on test set with best model...


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.30      0.43      0.35        30
        African National Congress (South Africa)       0.53      0.70      0.60        30
                                Al-Qaida in Iraq       0.43      0.60      0.50        30
        Al-Qaida in the Arabian Peninsula (AQAP)       0.41      0.23      0.30        30
                                      Al-Shabaab       0.23      0.17      0.19        30
             Basque Fatherland and Freedom (ETA)       0.45      0.67      0.54        30
                                      Boko Haram       0.38      0.17      0.23        30
  Communist Party of India - Maoist (CPI-Maoist)       0.47      0.63      0.54        30
       Corsican National Liberation Front (FLNC)       0.68      0.83      0.75        30
                       Donetsk People's Republic       0.41      0.47      0.44        30
Farabundo

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [ ]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true